<a href="https://colab.research.google.com/github/Zhang-Zhaoji/MIMIC-JAX-DEMO/blob/main/demos/rodent_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rodent demo

In this file you are able to try the stac-jax in google colab environment. Please make sure you are using a GPU runtime for we want to accelerate the algorithm.

Some of the package installed here may need you to restart the runtime. It's harmless and just do as what the colab said.

In [1]:
!git clone https://github.com/talmolab/stac-mjx.git
!mv stac-mjx/* ./
!nvidia-smi

Cloning into 'stac-mjx'...
remote: Enumerating objects: 2638, done.
remote: Counting objects: 100% (908/908), done.
remote: Compressing objects: 100% (402/402), done.
remote: Total 2638 (delta 699), reused 536 (delta 506), pack-reused 1730 (from 2)
Receiving objects: 100% (2638/2638), 167.11 MiB | 27.80 MiB/s, done.
Resolving deltas: 100% (1490/1490), done.


In [15]:
!uv pip install -e ".[cuda13]"
!pip uninstall --yes jaxlib jax jax-cuda12-pjrt jax-cuda12-plugin jax-cuda13-pjrt jax-cuda13-plugin
!pip install -U "jax[cuda13]"
!pip install warp-lang

Using Python 3.12.13 environment at: /usr
Resolved 122 packages in 998ms
Prepared 1 package in 246ms
Uninstalled 1 package in 0.49ms
Installed 1 package in 2ms
 ~ stac-mjx==0.0.1 (from file:///content)
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jax-cuda12-pjrt 0.7.2
Uninstalling jax-cuda12-pjrt-0.7.2:
  Successfully uninstalled jax-cuda12-pjrt-0.7.2
Found existing installation: jax-cuda12-plugin 0.7.2
Uninstalling jax-cuda12-plugin-0.7.2:
  Successfully uninstalled jax-cuda12-plugin-0.7.2
Found existing installation: jax-cuda13-pjrt 0.7.2
Uninstalling jax-cuda13-pjrt-0.7.2:
  Successfully uninstalled jax-cuda13-pjrt-0.7.2
Found existing installation: jax-cuda13-plugin 0.7.2
Uninstalling jax-cuda13-plugin-0.7.2:
  Successfully uninstalled jax-cuda13-plugin-0.7.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# test if installed properly
!python -c "import jax; print(f'JAX version: {jax.__version__}'); print(f'Available devices: {jax.devices()}')"

JAX version: 0.10.0
Available devices: [CudaDevice(id=0)]


In [2]:
!sudo apt-get update
!sudo apt-get install build-essential libgl1-mesa-dev libglu1-mesa-dev freeglut3-dev mesa-utils
!sudo apt install mesa-utils mesa-dri-drivers
!sudo apt install mesa-common-dev libgl1-mesa-dev libglu1-mesa-dev
!glxinfo | grep -i 'opengl version\|renderer'

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,064 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [8]:
!pip install --upgrade PyOpenGL PyOpenGL_accelerate
!sudo apt-get install libgl1-mesa-dev libglu1-mesa-dev freeglut3-dev
!pip install --upgrade pyrender

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libglu1-mesa-dev is already the newest version (9.0.2-1).
freeglut3-dev is already the newest version (2.8.1-6).
libgl1-mesa-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.4 MB/s eta 0:00:00
  Created wheel for PyOpenGL: filename=PyOpenGL-3.1.0-py3-none-any.whl size=1745193 sha256=8ccb76bf6ae30f01bfefc8e64af1d3cf698f9f21909b724a19ae0677b2b9c960
  Stored in directory: /root/.cache/pip/wheels/5c/d0/77/e69597cdbcb72ea27345036f549a737909bcf17c39789472ce
Successfully built PyOpenGL
  Attempt

In [13]:
# Use glfw or egl for faster rendering if your system supports it
import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

import stac_mjx
from pathlib import Path
import mediapy as media

base_path = Path.cwd()

### Load configs

In [14]:
print(base_path)
cfg = stac_mjx.load_configs(base_path / "configs")
cfg

/content
Config loaded and validated.


{'model': {'MJCF_PATH': 'models/rodent.xml', 'FTOL': 0.0001, 'ROOT_FTOL': 1e-05, 'LIMB_FTOL': 1e-06, 'N_ITERS': 6, 'N_ITER_Q': 400, 'KP_NAMES': ['Snout', 'EarL', 'EarR', 'SpineF', 'SpineM', 'SpineL', 'TailBase', 'ShoulderL', 'ElbowL', 'WristL', 'HandL', 'ShoulderR', 'ElbowR', 'WristR', 'HandR', 'HipL', 'KneeL', 'AnkleL', 'FootL', 'HipR', 'KneeR', 'AnkleR', 'FootR'], 'KEYPOINT_MODEL_PAIRS': {'AnkleL': 'lower_leg_L', 'AnkleR': 'lower_leg_R', 'EarL': 'skull', 'EarR': 'skull', 'ElbowL': 'upper_arm_L', 'ElbowR': 'upper_arm_R', 'FootL': 'foot_L', 'FootR': 'foot_R', 'HandL': 'hand_L', 'HandR': 'hand_R', 'HipL': 'pelvis', 'HipR': 'pelvis', 'KneeL': 'upper_leg_L', 'KneeR': 'upper_leg_R', 'ShoulderL': 'scapula_L', 'ShoulderR': 'scapula_R', 'Snout': 'skull', 'SpineF': 'vertebra_cervical_5', 'SpineL': 'pelvis', 'SpineM': 'vertebra_1', 'TailBase': 'pelvis', 'WristL': 'lower_arm_L', 'WristR': 'lower_arm_R'}, 'KEYPOINT_INITIAL_OFFSETS': {'AnkleL': '-0.03230154033783406 -0.004727054683585472 -0.022059

### Prepare your data

When using your own data, make sure the `kp_data` array is of the shape (n_frames, 3 * n_keypoints); note that the (xyz) axis is flattened into the keypoint dimension. `sorted_kp_names` contains an ordered list of each keypoint's name, as used in the config's `model.KEYPOINT_MODEL_PAIRS` field.

In [15]:
kp_data, sorted_kp_names = stac_mjx.load_data(cfg, base_path)

### Run stac!

In [ ]:
fit_path, ik_only_path = stac_mjx.run_stac(
    cfg,
    kp_data,
    sorted_kp_names,
    base_path=base_path
)
# this cell would run for around 1 hour on a t4 GPU...
# the output is like:
"""
Running fit. Mocap data shape: (10, 69)
Root Optimization:
Optimizing first 7 qposes for root optimization
Root optimization finished in 9.00 minutes with an error of 9.280361700803041e-05
Calibration iteration: 1/6
Pose Optimization:
"""

Running fit. Mocap data shape: (10, 69)
Root Optimization:
Optimizing first 7 qposes for root optimization
Root optimization finished in 9.00 minutes with an error of 9.280361700803041e-05
Calibration iteration: 1/6
Pose Optimization:
Pose Optimization finished in 4.67 minutes
Mean: 2.668361776159145e-05
Standard deviation: 1.1682639524224214e-05
Begining offset optimization:
Final residual error of 0.0004977943608537316
Offset optimization finished in 100.36087012290955 seconds
Calibration iteration: 2/6
Pose Optimization:
Pose Optimization finished in 0.18 minutes
Mean: 1.1157526387250982e-05
Standard deviation: 7.229302354971878e-06
Begining offset optimization:
Final residual error of 7.998878572834656e-06
Offset optimization finished in 0.0239713191986084 seconds
Calibration iteration: 3/6
Pose Optimization:
Pose Optimization finished in 0.17 minutes
Mean: 1.617501584405545e-05
Standard deviation: 7.220114184747217e-06
Begining offset optimization:
Final residual error of 4.528711

### Let's visualize!
It's only 10 frames at 50hz, so there's little movement, but you can see that the pose is now aligned to the keypoints!

In [ ]:
data_path = base_path / "demo_fit_offsets.h5"
n_frames = 10
save_path = base_path / "videos/direct_render.mp4"

# Call mujoco_viz
cfg, frames = stac_mjx.viz_stac(
   data_path,
   n_frames,
   save_path,
   start_frame=0,
   camera="close_profile",
   base_path=Path.cwd().parent,
)

# Show the video in the notebook (it is also saved to the save_path)
media.show_video(frames, fps=cfg.model.RENDER_FPS)

0it [00:00, ?it/s]/Users/charles/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
10it [00:00, 14.95it/s]


This browser does not support the video tag.
